# 00 Run Everything - Publication Workflow

This is the single notebook entry point. Run this first for the complete Coswara pipeline and publication extras. The other notebooks are optional review/debug notebooks only.

Default behavior is conservative: expensive or optional stages are controlled by toggles, and existing outputs are skipped unless `FORCE_REBUILD = True`.

In [1]:
from pathlib import Path
import os
import subprocess
import sys
from datetime import datetime

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "covid_audio_btp").exists():
            return candidate
    raise FileNotFoundError("Could not find project root. Start Jupyter from the extracted covid_audio_btp folder or one of its subfolders.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")
print(f"Started: {datetime.now().isoformat(timespec='seconds')}")

Project root: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp
Python: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python
Started: 2026-06-12T14:40:18


## Configuration

Set only the paths and toggles you need. Keep `RUN_CNN = False` until the classical baseline pipeline is clean.

In [10]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent

RAW_COSWARA_DIR = PROJECT_ROOT / "data/raw/coswara"
COUGHVID_RAW = PROJECT_ROOT / "data/raw/coughvid/v3_extracted"

FORCE_REBUILD = True

RUN_ENV_CHECK = True
RUN_LAYOUT_AUDIT = False

RUN_CORE_COSWARA = False
RUN_FEATURES = False
RUN_ML_BASELINES = False
RUN_CALIBRATION = False
RUN_FUSION = False
RUN_CNN = False
RUN_SHIFT_CHECKS = False
RUN_REPORT_ASSETS = False

RUN_PUBLICATION_EXTRAS = True
RUN_METADATA_BASELINE = False
RUN_QUALITY_WEIGHTED_FUSION = False
RUN_ABSTENTION = False
RUN_BOOTSTRAP_CI = False

RUN_COUGHVID_INDEX = False
RUN_COUGHVID_FEATURES = True
RUN_CROSS_DATASET = True
RUN_FEATURE_SHIFT_REPORT = True

RUN_PAPER_TABLES = True
RUN_PAIRED_MODEL_COMPARISON = False
RUN_CONFOUNDING_MATCHING = False
RUN_EXPERIMENT_MANIFEST = True

MIN_COUGH_DETECTED = 0.80
COUGHVID_FEATURE_MAX_ROWS = None

CNN_EPOCHS = 50
CNN_BATCH_SIZE = 8

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Coswara:", RAW_COSWARA_DIR)
print("COUGHVID:", COUGHVID_RAW)
print("Coswara exists:", RAW_COSWARA_DIR.exists())
print("COUGHVID exists:", COUGHVID_RAW.exists())

PROJECT_ROOT: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp
Coswara: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/data/raw/coswara
COUGHVID: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/data/raw/coughvid/v3_extracted
Coswara exists: True
COUGHVID exists: True


## Runner

The helper below runs project scripts, checks required inputs, and skips completed outputs unless forced.

In [11]:
def p(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def path_exists(path):
    return p(path).exists() and p(path).stat().st_size > 0


def missing(paths):
    return [str(path) for path in paths if not path_exists(path)]


def run_step(name, args, enabled=True, requires=None, creates=None, strict_requires=True, force=None):
    requires = requires or []
    creates = creates or []
    force = FORCE_REBUILD if force is None else force
    if not enabled:
        print(f"SKIP {name}: disabled")
        return False
    missing_inputs = missing(requires)
    if missing_inputs:
        message = f"SKIP {name}: missing required input(s): {missing_inputs}"
        if strict_requires:
            raise FileNotFoundError(message)
        print(message)
        return False
    if creates and not force and all(path_exists(path) for path in creates):
        print(f"SKIP {name}: outputs already exist")
        return False
    cmd = [str(item) for item in args]
    print()
    print(f"## {name}")
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
    return True


CORE_ARTIFACTS = [
    "data/interim/coswara_index.csv",
    "data/processed/metadata_clean.csv",
    "data/interim/modality_availability.csv",
    "data/interim/split_manifest.csv",
    "data/processed/audio_quality.csv",
    "data/processed/features_mfcc.csv",
    "data/processed/spectrogram_index.csv",
    "data/outputs/metrics/ml_baseline_metrics.csv",
    "data/outputs/metrics/calibrated_branch_predictions.csv",
    "data/outputs/metrics/fusion_metrics.csv",
    "data/outputs/metrics/quality_weighted_fusion_metrics.csv",
    "reports/tables/paper_metric_table.csv",
    "data/outputs/metrics/paired_model_comparison.csv",
    "reports/tables/confounding_balance.csv",
    "reports/tables/feature_shift_report.csv",
    "reports/experiment_manifest.json",
]
print("Runner ready")

Runner ready


## Artifact Dashboard

This shows what already exists before the run.

In [12]:
try:
    import pandas as pd
    dashboard = pd.DataFrame(
        [{"path": path, "exists": path_exists(path), "size_bytes": p(path).stat().st_size if p(path).exists() else 0} for path in CORE_ARTIFACTS]
    )
    display(dashboard)
except Exception as exc:
    print(f"Dashboard unavailable before dependency install: {exc}")
    for path in CORE_ARTIFACTS:
        print(path, "OK" if path_exists(path) else "missing")

,path,exists,size_bytes
0,data/interim/coswara_index.csv,True,14467114
1,data/processed/metadata_clean.csv,True,15192524
2,data/interim/modality_availability.csv,True,211550
3,data/interim/split_manifest.csv,True,243621
4,data/processed/audio_quality.csv,True,8643031
5,data/processed/features_mfcc.csv,True,90512724
6,data/processed/spectrogram_index.csv,True,3879710
7,data/outputs/metrics/ml_baseline_metrics.csv,True,2362
8,data/outputs/metrics/calibrated_branch_predict...,True,1480378
9,data/outputs/metrics/fusion_metrics.csv,True,538


## Stage 0-1: Environment, Index, Splits, Quality

This stage is mandatory before modeling. It prevents training on bad audio, bad labels, or participant leakage.

In [13]:
run_step("local preflight", [sys.executable, "scripts/00_local_preflight.py", "--coswara-dir", RAW_COSWARA_DIR], enabled=True, force=True)

run_step("environment check", [sys.executable, "scripts/00_check_environment.py"], enabled=RUN_ENV_CHECK, force=True)

if RUN_CORE_COSWARA and not RAW_COSWARA_DIR.exists():
    raise FileNotFoundError(f"Coswara not found. Put it at {RAW_COSWARA_DIR} or change RAW_COSWARA_DIR.")

run_step(
    "raw layout audit",
    [sys.executable, "scripts/00_inspect_dataset_layout.py", "--raw-dir", RAW_COSWARA_DIR, "--output", "reports/tables/coswara_layout_audit.csv"],
    enabled=RUN_CORE_COSWARA and RUN_LAYOUT_AUDIT,
    creates=["reports/tables/coswara_layout_audit.csv"],
)
run_step(
    "build Coswara index",
    [sys.executable, "scripts/01_build_coswara_index.py", "--raw-dir", RAW_COSWARA_DIR, "--output", "data/interim/coswara_index.csv"],
    enabled=RUN_CORE_COSWARA,
    creates=["data/interim/coswara_index.csv"],
)
run_step(
    "clean metadata",
    [sys.executable, "scripts/02_clean_metadata.py", "--index", "data/interim/coswara_index.csv", "--output", "data/processed/metadata_clean.csv", "--availability-output", "data/interim/modality_availability.csv", "--audit-output", "reports/tables/dataset_audit.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/coswara_index.csv"],
    creates=["data/processed/metadata_clean.csv", "data/interim/modality_availability.csv"],
)
run_step(
    "participant splits",
    [sys.executable, "scripts/03_create_splits.py", "--metadata", "data/processed/metadata_clean.csv", "--output", "data/interim/split_manifest.csv", "--metadata-output", "data/processed/metadata_clean.csv", "--audit-output", "reports/tables/split_audit.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/interim/split_manifest.csv"],
)
run_step(
    "quality audit",
    [sys.executable, "scripts/04_quality_audit.py", "--metadata", "data/processed/metadata_clean.csv", "--output", "data/processed/audio_quality.csv", "--metadata-output", "data/processed/metadata_clean.csv", "--summary-output", "reports/tables/quality_summary.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/split_manifest.csv", "data/processed/metadata_clean.csv"],
    creates=["data/processed/audio_quality.csv"],
)
run_step(
    "artifact validation",
    [sys.executable, "scripts/12_validate_artifacts.py", "--strict"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/coswara_index.csv", "data/processed/metadata_clean.csv", "data/processed/audio_quality.csv"],
    force=True,
)


## local preflight
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/00_local_preflight.py --coswara-dir /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/data/raw/coswara
Project root: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp
Python: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python
Coswara path: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/data/raw/coswara
Notebook syntax: OK
Python syntax: OK
Required imports: OK
Coswara audio files discovered: 24712
Coswara CSV files discovered: 73

WARNINGS
- xgboost (xgboost): No module named 'xgboost'
- streamlit (streamlit): No module named 'streamlit'

Preflight passed. It is safe to start the notebook pipeline.


## environment check
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/00_check_environment.py
Python: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
OK required covid_audio_btp: 0.1.0
OK required numpy: 2.4.6
OK required pandas: 3.0.3
OK required scip

False

## Hard Gate

If this cell fails, inspect the optional review notebooks before training models.

In [14]:
import pandas as pd

from covid_audio_btp.notebook_utils import (
    assert_binary_labels_present,
    assert_no_participant_leakage,
    check_artifacts,
    read_optional_csv,
    stop_if_validation_errors,
)

metadata = pd.read_csv(p("data/processed/metadata_clean.csv"))
quality = pd.read_csv(p("data/processed/audio_quality.csv"))
issues = read_optional_csv("reports/tables/validation_issues.csv")

assert_binary_labels_present(metadata)
assert_no_participant_leakage(metadata)
if issues is not None:
    stop_if_validation_errors(issues)

quality_ok_rate = float((quality["quality_flag"] == "ok").mean()) if len(quality) else 0.0
print(f"quality ok rate: {quality_ok_rate:.3f}")
if quality_ok_rate < 0.50:
    raise AssertionError("Quality ok rate below 50%; review audio quality before modeling.")

reports/tables/validation_issues.csv: 1 rows x 3 columns
label gate passed: both positive and negative classes are present
leakage gate passed: 2746 participants appear in one split each
validation gate passed with warnings only
quality ok rate: 0.953


## Stage 2-5: Features, Baselines, Calibration, Fusion

This is the main Coswara modeling path. CNN is optional and disabled by default.

In [15]:
run_step(
    "feature extraction",
    [
        sys.executable,
        "scripts/05_extract_features.py",
        "--metadata",
        "data/processed/metadata_clean.csv",
        "--features-output",
        "data/processed/features_mfcc.csv",
        "--quality-mode",
        "all_samples",
        "--skip-spectrograms",
    ],
    enabled=RUN_FEATURES,
    requires=[
        "data/processed/metadata_clean.csv",
        "data/processed/audio_quality.csv",
    ],
    creates=["data/processed/features_mfcc.csv"],
    force=FORCE_REBUILD,
)
run_step(
    "classical ML baselines",
    [sys.executable, "scripts/06_train_ml_baselines.py", "--features", "data/processed/features_mfcc.csv"],
    enabled=RUN_ML_BASELINES,
    requires=["data/processed/features_mfcc.csv"],
    creates=["data/outputs/metrics/ml_baseline_metrics.csv", "data/outputs/metrics/ml_validation_metrics.csv", "data/outputs/metrics/ml_predictions_validation.csv", "data/outputs/metrics/ml_predictions_test.csv"],
)
run_step(
    "compact CNN cough branch",
    [sys.executable, "scripts/07_train_cnn.py", "--spectrogram-index", "data/processed/spectrogram_index.csv", "--modality", "cough", "--epochs", CNN_EPOCHS, "--batch-size", CNN_BATCH_SIZE],
    enabled=RUN_CNN,
    requires=["data/processed/spectrogram_index.csv"],
    creates=["data/outputs/metrics/cnn_metrics.csv", "data/outputs/metrics/cnn_logits_validation.csv", "data/outputs/metrics/cnn_logits_test.csv"],
)
run_step(
    "branch calibration",
    [sys.executable, "scripts/08_calibrate_branches.py", "--validation-predictions", "data/outputs/metrics/ml_predictions_validation.csv", "--test-predictions", "data/outputs/metrics/ml_predictions_test.csv", "--method", "platt"],
    enabled=RUN_CALIBRATION,
    requires=["data/outputs/metrics/ml_predictions_validation.csv", "data/outputs/metrics/ml_predictions_test.csv"],
    creates=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/outputs/metrics/calibration_metrics.csv"],
)
run_step(
    "standard fusion",
    [sys.executable, "scripts/09_run_fusion.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--validation-metrics", "data/outputs/metrics/ml_validation_metrics.csv"],
    enabled=RUN_FUSION,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/outputs/metrics/ml_validation_metrics.csv"],
    creates=["data/outputs/metrics/fusion_predictions.csv", "data/outputs/metrics/fusion_metrics.csv"],
)
run_step(
    "subgroup and confounding checks",
    [sys.executable, "scripts/10_shift_and_confounding_checks.py", "--predictions", "data/outputs/metrics/fusion_predictions.csv", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_SHIFT_CHECKS,
    requires=["data/outputs/metrics/fusion_predictions.csv", "data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/subgroup_metrics.csv"],
)
run_step(
    "basic report assets",
    [sys.executable, "scripts/11_make_report_assets.py", "--metadata", "data/processed/metadata_clean.csv", "--predictions", "data/outputs/metrics/fusion_predictions.csv"],
    enabled=RUN_REPORT_ASSETS,
    requires=["data/processed/metadata_clean.csv"],
    creates=["reports/report_outline.md", "reports/slides_outline.md"],
)

SKIP feature extraction: disabled
SKIP classical ML baselines: disabled
SKIP compact CNN cough branch: disabled
SKIP branch calibration: disabled
SKIP standard fusion: disabled
SKIP subgroup and confounding checks: disabled
SKIP basic report assets: disabled


False

## Stage 6: Publication Extras

These experiments support the publication-grade claims: metadata-only baseline, quality-weighted fusion, abstention, confidence intervals, COUGHVID external validation, and paper tables.

In [16]:
run_step(
    "metadata-only baseline",
    [sys.executable, "scripts/14_train_metadata_baseline.py", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_METADATA_BASELINE,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/metadata_baseline_metrics.csv", "data/outputs/metrics/metadata_predictions_validation.csv", "data/outputs/metrics/metadata_predictions_test.csv"],
)
run_step(
    "quality-weighted fusion",
    [sys.executable, "scripts/16_run_quality_weighted_fusion.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--quality", "data/processed/audio_quality.csv", "--validation-metrics", "data/outputs/metrics/ml_validation_metrics.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_QUALITY_WEIGHTED_FUSION,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/processed/audio_quality.csv", "data/outputs/metrics/ml_validation_metrics.csv"],
    creates=["data/outputs/metrics/quality_weighted_fusion_predictions.csv", "data/outputs/metrics/quality_weighted_fusion_metrics.csv"],
)
run_step(
    "abstention analysis",
    [sys.executable, "scripts/17_abstention_analysis.py", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_ABSTENTION,
    requires=["data/outputs/metrics/quality_weighted_fusion_predictions.csv", "data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/abstention_decisions.csv", "data/outputs/metrics/abstention_coverage_curve.csv"],
)
run_step(
    "bootstrap CI for quality-weighted fusion",
    [sys.executable, "scripts/15_bootstrap_prediction_metrics.py", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--output", "data/outputs/metrics/quality_weighted_fusion_bootstrap_ci.csv", "--group-columns", "fusion_method"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_BOOTSTRAP_CI,
    requires=["data/outputs/metrics/quality_weighted_fusion_predictions.csv"],
    creates=["data/outputs/metrics/quality_weighted_fusion_bootstrap_ci.csv"],
)

if COUGHVID_RAW is not None:
    run_step(
        "COUGHVID index",
        [sys.executable, "scripts/13_build_coughvid_index.py", "--raw-dir", COUGHVID_RAW, "--output", "data/interim/coughvid_index.csv", "--min-cough-detected", MIN_COUGH_DETECTED],
        enabled=RUN_PUBLICATION_EXTRAS and RUN_COUGHVID_INDEX,
        creates=["data/interim/coughvid_index.csv"],
    )
else:
    print("SKIP COUGHVID index: COUGHVID_RAW is None")

feature_cmd = [sys.executable, "scripts/19_extract_coughvid_features.py", "--index", "data/interim/coughvid_index.csv", "--features-output", "data/processed/coughvid_features_mfcc.csv", "--quality-ok-only"]
if COUGHVID_FEATURE_MAX_ROWS is not None:
    feature_cmd += ["--max-rows", COUGHVID_FEATURE_MAX_ROWS]
run_step(
    "COUGHVID feature extraction",
    feature_cmd,
    enabled=RUN_PUBLICATION_EXTRAS and RUN_COUGHVID_FEATURES,
    requires=["data/interim/coughvid_index.csv"],
    creates=["data/processed/coughvid_features_mfcc.csv"],
    strict_requires=False,
)
run_step(
    "cross-dataset cough evaluation",
    [sys.executable, "scripts/18_cross_dataset_feature_eval.py", "--source-features", "data/processed/features_mfcc.csv", "--external-features", "data/processed/coughvid_features_mfcc.csv", "--modality", "cough", "--model-name", "logistic_regression"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_CROSS_DATASET,
    requires=["data/processed/features_mfcc.csv", "data/processed/coughvid_features_mfcc.csv"],
    creates=["data/outputs/metrics/cross_dataset_predictions.csv", "data/outputs/metrics/cross_dataset_metrics.csv"],
    strict_requires=False,
)
run_step(
    "paper metric tables",
    [sys.executable, "scripts/20_make_paper_tables.py"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_PAPER_TABLES,
    requires=["data/outputs/metrics/ml_baseline_metrics.csv"],
    creates=["reports/tables/paper_metric_table.csv", "reports/tables/paper_metric_table_raw.csv"],
    strict_requires=False,
)

SKIP metadata-only baseline: disabled
SKIP quality-weighted fusion: disabled
SKIP abstention analysis: disabled
SKIP bootstrap CI for quality-weighted fusion: disabled
SKIP COUGHVID index: disabled

## COUGHVID feature extraction
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/19_extract_coughvid_features.py --index data/interim/coughvid_index.csv --features-output data/processed/coughvid_features_mfcc.csv --quality-ok-only


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Wrote cross-dataset predictions: data/outputs/metrics/cross_dataset_predictions.csv
Wrote cross-dataset metrics: data/outputs/metrics/cross_dataset_metrics.csv
  auroc    auprc  balanced_accuracy       f1  sensitivity  specificity    brier      ece      nll  threshold  n_samples          model_name modality  source_rows  external_rows  n_features
0.52948 0.043007           0.522575 0.071313     0.231579     0.813572 0.151196 0.279755 0.472138        0.5     8331.0 logistic_regression    cough         2957           8331         253


## paper metric tables
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/20_make_paper_tables.py
Wrote paper metric table: reports/tables/paper_metric_table.csv (30 rows)
Wrote raw combined metric table: reports/tables/paper_metric_table_raw.csv (30 rows)



True

In [20]:
import subprocess
import sys
import pandas as pd
import numpy as np

cmds = [
  [
      sys.executable,
      "scripts/23_feature_shift_report.py",
      "--source-features",
      "data/processed/features_mfcc.csv",
      "--external-features",
      "data/processed/coughvid_features_mfcc.csv",
      "--output",
      "reports/tables/feature_shift_report.csv",
      "--summary-output",
      "reports/tables/feature_shift_summary.csv",
  ],
  [
      sys.executable,
      "scripts/24_make_experiment_manifest.py",
  ],
  [
      sys.executable,
      "scripts/12_validate_artifacts.py",
      "--strict",
  ],
]

for cmd in cmds:
  print("\n$", " ".join(cmd))
  result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
  print(result.stdout)
  print(result.stderr)
  result.check_returncode()




$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/23_feature_shift_report.py --source-features data/processed/features_mfcc.csv --external-features data/processed/coughvid_features_mfcc.csv --output reports/tables/feature_shift_report.csv --summary-output reports/tables/feature_shift_summary.csv
Wrote feature shift report: reports/tables/feature_shift_report.csv (253 rows)
Wrote feature shift summary: reports/tables/feature_shift_summary.csv



$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/24_make_experiment_manifest.py
Wrote experiment manifest: reports/experiment_manifest.json



$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/12_validate_artifacts.py --strict
severity          check                       message
 warning unknown_labels 5688 rows have unknown labels




In [21]:
feat = pd.read_csv(PROJECT_ROOT / "data/processed/coughvid_features_mfcc.csv")
cross = pd.read_csv(PROJECT_ROOT / "data/outputs/metrics/cross_dataset_metrics.csv")
shift_summary = pd.read_csv(PROJECT_ROOT / "reports/tables/feature_shift_summary.csv")

print("Full COUGHVID feature shape:", feat.shape)

print("\nFeature labels:")
print(feat["label_binary"].value_counts(dropna=False))

print("\nNaN count:", int(feat.select_dtypes(include="number").isna().sum().sum()))
print("Inf count:", int(np.isinf(feat.select_dtypes(include="number").to_numpy()).sum()))

print("\nCross-dataset metrics:")
display(cross)

print("\nFeature shift summary:")
display(shift_summary)

Full COUGHVID feature shape: (8331, 261)

Feature labels:
label_binary
negative    8046
positive     285
Name: count, dtype: int64

NaN count: 0
Inf count: 0

Cross-dataset metrics:


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,brier,ece,nll,threshold,n_samples,model_name,modality,source_rows,external_rows,n_features
0,0.52948,0.043007,0.522575,0.071313,0.231579,0.813572,0.151196,0.279755,0.472138,0.5,8331.0,logistic_regression,cough,2957,8331,253



Feature shift summary:


,n_features,n_high_shift_features,max_abs_standardized_mean_difference,smd_threshold
0,253,21,2.645905,0.5


In [9]:
import numpy as np
import pandas as pd

idx = pd.read_csv(PROJECT_ROOT / "data/interim/coughvid_index.csv")
feat = pd.read_csv(PROJECT_ROOT / "data/processed/coughvid_features_mfcc.csv")

print("Index shape:", idx.shape)
print("Smoke feature shape:", feat.shape)

print("\nIndex labels:")
print(idx["label_binary"].value_counts(dropna=False))

print("\nSmoke labels:")
print(feat["label_binary"].value_counts(dropna=False))

numeric = feat.select_dtypes(include="number")
print("\nNaN count:", int(numeric.isna().sum().sum()))
print("Inf count:", int(np.isinf(numeric.to_numpy()).sum()))

assert feat.shape[0] == 25, "Smoke extraction did not produce 25 rows"
assert feat.shape[1] == 261, "Unexpected feature column count"
assert numeric.isna().sum().sum() == 0, "NaNs found in features"
assert np.isinf(numeric.to_numpy()).sum() == 0, "Inf values found in features"

print("\nCOUGHVID smoke test is clean.")

Index shape: (18544, 22)
Smoke feature shape: (25, 261)

Index labels:
label_binary
unknown     10213
negative     8046
positive      285
Name: count, dtype: int64

Smoke labels:
label_binary
negative    25
Name: count, dtype: int64

NaN count: 0
Inf count: 0

COUGHVID smoke test is clean.


In [23]:
from pathlib import Path
import shutil
import zipfile

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent

bundle_dir = PROJECT_ROOT.parent / "Publication_ExternalValidation_Artifacts_2026-06-12"
zip_path = PROJECT_ROOT.parent / "Publication_ExternalValidation_Artifacts_2026-06-12.zip"

if bundle_dir.exists():
  shutil.rmtree(bundle_dir)

(bundle_dir / "metrics").mkdir(parents=True)
(bundle_dir / "tables").mkdir(parents=True)
(bundle_dir / "reports").mkdir(parents=True)

files_to_copy = [
  ("data/interim/coughvid_index.csv", "tables/coughvid_index.csv"),
  ("data/processed/coughvid_features_mfcc.csv", "tables/coughvid_features_mfcc.csv"),
  ("data/outputs/metrics/cross_dataset_metrics.csv", "metrics/cross_dataset_metrics.csv"),
  ("data/outputs/metrics/cross_dataset_predictions.csv", "metrics/cross_dataset_predictions.csv"),
  ("reports/tables/feature_shift_report.csv", "tables/feature_shift_report.csv"),
  ("reports/tables/feature_shift_summary.csv", "tables/feature_shift_summary.csv"),
  ("reports/tables/paper_metric_table.csv", "tables/paper_metric_table.csv"),
  ("reports/tables/paper_metric_table_raw.csv", "tables/paper_metric_table_raw.csv"),
  ("reports/experiment_manifest.json", "reports/experiment_manifest.json"),
]

for src_rel, dst_rel in files_to_copy:
  src = PROJECT_ROOT / src_rel
  dst = bundle_dir / dst_rel
  if src.exists():
      shutil.copy2(src, dst)
      print("copied:", src_rel)
  else:
      print("missing:", src_rel)

readme = bundle_dir / "README.md"
readme.write_text(
  """# Publication External Validation Artifacts

Date: 2026-06-12
Scope: COUGHVID v3 external validation after corrected no-leakage Coswara baseline.

Completed:
- COUGHVID v3 extraction
- COUGHVID index
- COUGHVID MFCC feature extraction
- Coswara-to-COUGHVID cough-only external validation
- Feature-shift report
- Paper metric table refresh
- Experiment manifest refresh

Key result:
- Logistic regression cough-only Coswara-to-COUGHVID AUROC: 0.52948
- AUPRC: 0.043007
- Balanced accuracy: 0.522575
- F1: 0.071313
- External labeled rows: 8331
- High-shift features: 21 / 253

Interpretation:
This is evidence of strong cross-dataset shift. It should be framed as an external stress test, not a failed implementation.
""",
  encoding="utf-8",
)

if zip_path.exists():
  zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
  for file in bundle_dir.rglob("*"):
      z.write(file, file.relative_to(bundle_dir.parent))

print("\nBundle folder:", bundle_dir)
print("Bundle zip:", zip_path)
print("Zip exists:", zip_path.exists())
print("Zip size MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))

copied: data/interim/coughvid_index.csv
copied: data/processed/coughvid_features_mfcc.csv
copied: data/outputs/metrics/cross_dataset_metrics.csv
copied: data/outputs/metrics/cross_dataset_predictions.csv
copied: reports/tables/feature_shift_report.csv
copied: reports/tables/feature_shift_summary.csv
copied: reports/tables/paper_metric_table.csv
copied: reports/tables/paper_metric_table_raw.csv
copied: reports/experiment_manifest.json

Bundle folder: /home/covid/Desktop/Covid-19-BTP/Publication_ExternalValidation_Artifacts_2026-06-12
Bundle zip: /home/covid/Desktop/Covid-19-BTP/Publication_ExternalValidation_Artifacts_2026-06-12.zip
Zip exists: True
Zip size MB: 17.58


## Stage 7: Extra Publication Strengthening

These optional diagnostics make the paper stronger: paired model comparison, matched confounding subset, source-vs-external feature shift, and a reproducibility manifest.

In [ ]:
run_step(
    "paired ML model comparison",
    [sys.executable, "scripts/21_paired_model_comparison.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--baseline-name", "logistic_regression", "--model-column", "model_name", "--group-columns", "modality", "--output", "data/outputs/metrics/paired_model_comparison.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_PAIRED_MODEL_COMPARISON,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv"],
    creates=["data/outputs/metrics/paired_model_comparison.csv"],
    strict_requires=False,
)
run_step(
    "confounding matched subset",
    [sys.executable, "scripts/22_confounding_matching.py", "--metadata", "data/processed/metadata_clean.csv", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--covariates", "age_bucket", "gender"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_CONFOUNDING_MATCHING,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/processed/metadata_matched.csv", "reports/tables/confounding_balance.csv"],
    strict_requires=False,
)
run_step(
    "feature shift report",
    [sys.executable, "scripts/23_feature_shift_report.py", "--source-features", "data/processed/features_mfcc.csv", "--external-features", "data/processed/coughvid_features_mfcc.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_FEATURE_SHIFT_REPORT,
    requires=["data/processed/features_mfcc.csv", "data/processed/coughvid_features_mfcc.csv"],
    creates=["reports/tables/feature_shift_report.csv", "reports/tables/feature_shift_summary.csv"],
    strict_requires=False,
)
run_step(
    "experiment manifest",
    [sys.executable, "scripts/24_make_experiment_manifest.py", "--run-name", "covid_audio_publication_run"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_EXPERIMENT_MANIFEST,
    creates=["reports/experiment_manifest.json"],
)

## Final Dashboard

Use this at the end of the run to see exactly what was produced.

In [ ]:
import pandas as pd

final_dashboard = pd.DataFrame(
    [{"path": path, "exists": path_exists(path), "size_bytes": p(path).stat().st_size if p(path).exists() else 0} for path in CORE_ARTIFACTS]
)
display(final_dashboard)

for path in [
    "data/outputs/metrics/ml_baseline_metrics.csv",
    "data/outputs/metrics/fusion_metrics.csv",
    "data/outputs/metrics/quality_weighted_fusion_metrics.csv",
    "reports/tables/paper_metric_table.csv",
    "data/outputs/metrics/paired_model_comparison.csv",
    "reports/tables/confounding_balance.csv",
    "reports/tables/feature_shift_report.csv",
    "reports/experiment_manifest.json",
]:
    if path_exists(path):
        print()
        print(path)
        display(pd.read_csv(p(path)).head(20))

print("Run finished")

In [1]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

def run_cmd(name, args):
  print(f"\n## {name}")
  print("$", " ".join(str(a) for a in args))
  subprocess.run([str(a) for a in args], check=True)

required_files = [
  "data/processed/features_mfcc.csv",
  "data/processed/coughvid_features_mfcc.csv",
  "reports/tables/feature_shift_report.csv",
  "data/outputs/metrics/ml_validation_metrics.csv",
]

missing = [p for p in required_files if not Path(p).exists()]
if missing:
  raise FileNotFoundError(f"Missing required files. Stop here: {missing}")

run_cmd("install optional rescue dependencies", [sys.executable, "-m", "pip", "install", "-r", "requirements-optional.txt"])
run_cmd("environment check", [sys.executable, "scripts/00_check_environment.py"])

print("\nReady for rescue experiments.")


## install optional rescue dependencies
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python -m pip install -r requirements-optional.txt
  Using cached xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl.metadata (2.1 kB)
  Using cached streamlit-1.58.0-py3-none-any.whl.metadata (9.6 kB)
  Using cached altair-6.2.1-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached cachetools-7.1.4-py3-none-any.whl.metadata (5.5 kB)
  Using cached click-8.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached gitpython-3.1.50-py3-none-any.whl.metadata (14 kB)
  Using cached pydeck-0.9.2-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached uvicorn-0.49.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached httptools-0.8

In [2]:
run_cmd("focused rescue tests", [sys.executable, "-m", "pytest", "tests/test_rescue_experiments.py", "-q"])
run_cmd("full test suite", [sys.executable, "-m", "pytest", "tests", "-q"])


## focused rescue tests
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python -m pytest tests/test_rescue_experiments.py -q
.....                                                                    [100%]

## full test suite
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python -m pytest tests -q
.........................................................                [100%]
=============================== warnings summary ===============================
tests/test_publication_layer.py::test_abstention_marks_uncertain_and_low_quality_samples
tests/test_publication_layer.py::test_abstention_marks_uncertain_and_low_quality_samples
tests/test_publication_layer.py::test_abstention_marks_uncertain_and_low_quality_samples
tests/test_publication_layer.py::test_abstention_marks_uncertain_and_low_quality_samples
tests/test_publication_layer.py::test_abstention_marks_uncertain_and_low_quality_samples
tests/test_publication_layer.py::test_abstention_marks_uncertain_and

In [3]:
run_cmd(
  "COUGHVID internal baseline smoke",
  [
      sys.executable,
      "scripts/26_run_coughvid_internal_baseline.py",
      "--features", "data/processed/coughvid_features_mfcc.csv",
      "--models", "logistic_regression", "random_forest",
      "--split-features-output", "data/processed/coughvid_features_mfcc_internal_split_smoke.csv",
      "--predictions-output", "data/outputs/metrics/coughvid_internal_predictions_smoke.csv",
      "--metrics-output", "data/outputs/metrics/coughvid_internal_metrics_smoke.csv",
      "--bootstrap-output", "data/outputs/metrics/coughvid_internal_bootstrap_ci_smoke.csv",
      "--n-bootstraps", "50",
  ],
)

run_cmd(
  "external model grid smoke",
  [
      sys.executable,
      "scripts/25_run_external_model_grid.py",
      "--source-features", "data/processed/features_mfcc.csv",
      "--external-features", "data/processed/coughvid_features_mfcc.csv",
      "--feature-shift-report", "reports/tables/feature_shift_report.csv",
      "--models", "logistic_regression", "random_forest",
      "--feature-strategies", "all", "drop_high_shift", "top_stable_50",
      "--predictions-output", "data/outputs/metrics/external_model_grid_predictions_smoke.csv",
      "--metrics-output", "data/outputs/metrics/external_model_grid_metrics_smoke.csv",
      "--bootstrap-output", "data/outputs/metrics/external_model_grid_bootstrap_ci_smoke.csv",
      "--n-bootstraps", "50",
  ],
)


## COUGHVID internal baseline smoke
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/26_run_coughvid_internal_baseline.py --features data/processed/coughvid_features_mfcc.csv --models logistic_regression random_forest --split-features-output data/processed/coughvid_features_mfcc_internal_split_smoke.csv --predictions-output data/outputs/metrics/coughvid_internal_predictions_smoke.csv --metrics-output data/outputs/metrics/coughvid_internal_metrics_smoke.csv --bootstrap-output data/outputs/metrics/coughvid_internal_bootstrap_ci_smoke.csv --n-bootstraps 50
Ran COUGHVID internal logistic_regression: AUROC=0.7144, AUPRC=0.1376, threshold=0.0328
Ran COUGHVID internal random_forest: AUROC=0.7670, AUPRC=0.1199, threshold=0.0345
Wrote COUGHVID split features: data/processed/coughvid_features_mfcc_internal_split_smoke.csv (8331 rows)
Wrote COUGHVID internal predictions: data/outputs/metrics/coughvid_internal_predictions_smoke.csv (3334 rows)
Wrote COUGHVID internal me

In [4]:
run_cmd(
  "COUGHVID internal baseline final",
  [
      sys.executable,
      "scripts/26_run_coughvid_internal_baseline.py",
      "--features", "data/processed/coughvid_features_mfcc.csv",
      "--n-bootstraps", "1000",
  ],
)

run_cmd(
  "external model grid final",
  [
      sys.executable,
      "scripts/25_run_external_model_grid.py",
      "--source-features", "data/processed/features_mfcc.csv",
      "--external-features", "data/processed/coughvid_features_mfcc.csv",
      "--feature-shift-report", "reports/tables/feature_shift_report.csv",
      "--n-bootstraps", "1000",
  ],
)

run_cmd("paper metric tables", [sys.executable, "scripts/20_make_paper_tables.py"])
run_cmd("experiment manifest", [sys.executable, "scripts/24_make_experiment_manifest.py"])
run_cmd("artifact validation", [sys.executable, "scripts/12_validate_artifacts.py", "--strict"])


## COUGHVID internal baseline final
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/26_run_coughvid_internal_baseline.py --features data/processed/coughvid_features_mfcc.csv --n-bootstraps 1000
Ran COUGHVID internal logistic_regression: AUROC=0.7144, AUPRC=0.1376, threshold=0.0328
Ran COUGHVID internal random_forest: AUROC=0.7670, AUPRC=0.1199, threshold=0.0345
Ran COUGHVID internal xgboost: AUROC=0.7542, AUPRC=0.1379, threshold=0.0302
Ran COUGHVID internal lightgbm: AUROC=0.7809, AUPRC=0.1778, threshold=0.0340
Ran COUGHVID internal catboost: AUROC=0.7522, AUPRC=0.1341, threshold=0.0312
Wrote COUGHVID split features: data/processed/coughvid_features_mfcc_internal_split.csv (8331 rows)
Wrote COUGHVID internal predictions: data/outputs/metrics/coughvid_internal_predictions.csv (8335 rows)
Wrote COUGHVID internal metrics: data/outputs/metrics/coughvid_internal_metrics.csv (5 rows)
Wrote COUGHVID internal bootstrap CIs: data/outputs/metrics/coughvid_internal_bo

In [5]:
outputs = [
  "data/outputs/metrics/coughvid_internal_metrics.csv",
  "data/outputs/metrics/coughvid_internal_bootstrap_ci.csv",
  "data/outputs/metrics/external_model_grid_metrics.csv",
  "data/outputs/metrics/external_model_grid_bootstrap_ci.csv",
  "reports/tables/paper_metric_table.csv",
  "reports/experiment_manifest.json",
]

for p in outputs:
  path = Path(p)
  print(p, "exists=", path.exists(), "size=", path.stat().st_size if path.exists() else 0)

print("\nCOUGHVID internal baseline:")
display(pd.read_csv("data/outputs/metrics/coughvid_internal_metrics.csv").sort_values(["auroc", "auprc"], ascending=False))

print("\nExternal Coswara-to-COUGHVID model grid:")
display(pd.read_csv("data/outputs/metrics/external_model_grid_metrics.csv").sort_values(["auroc", "auprc"], ascending=False))

print("\nPaper table tail:")
display(pd.read_csv("reports/tables/paper_metric_table.csv").tail(40))

data/outputs/metrics/coughvid_internal_metrics.csv exists= True size= 1499
data/outputs/metrics/coughvid_internal_bootstrap_ci.csv exists= True size= 2332
data/outputs/metrics/external_model_grid_metrics.csv exists= True size= 6598
data/outputs/metrics/external_model_grid_bootstrap_ci.csv exists= True size= 12470
reports/tables/paper_metric_table.csv exists= True size= 6521
reports/experiment_manifest.json exists= True size= 3962

COUGHVID internal baseline:


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,brier,ece,nll,threshold,n_samples,model_name,modality,feature_strategy,calibration_method,dataset,train_rows,validation_rows,test_rows,n_features
3,0.780931,0.177806,0.697782,0.179775,0.561404,0.834161,0.032850,0.000060,0.147027,0.033965,1667.0,lightgbm,cough,all,platt,coughvid,4998,1666,1667,253
1,0.767015,0.119891,0.714934,0.165548,0.649123,0.780745,0.032947,0.000092,0.147902,0.034488,1667.0,random_forest,cough,all,platt,coughvid,4998,1666,1667,253
2,0.754223,0.137874,0.685507,0.133100,0.666667,0.704348,0.031948,0.002515,0.138600,0.030180,1667.0,xgboost,cough,all,platt,coughvid,4998,1666,1667,253
4,0.752163,0.134136,0.679443,0.114883,0.771930,0.586957,0.032414,0.000609,0.142485,0.031215,1667.0,catboost,cough,all,platt,coughvid,4998,1666,1667,253
0,0.714416,0.137571,0.617435,0.121951,0.438596,0.796273,0.032042,0.005884,0.139350,0.032802,1667.0,logistic_regression,cough,all,platt,coughvid,4998,1666,1667,253



External Coswara-to-COUGHVID model grid:


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,brier,ece,nll,threshold,n_samples,model_name,modality,feature_strategy,calibration_method,source_rows,validation_rows,external_rows,n_features
2,0.534794,0.042176,0.531858,0.072419,0.494737,0.568978,0.115092,0.284512,0.410796,0.320782,8331.0,logistic_regression,cough,top_stable_50,platt,2957,634,8331,50
20,0.526301,0.039278,0.517284,0.068203,0.259649,0.774919,0.104052,0.231047,0.365279,0.355506,8331.0,catboost,cough,all,platt,2957,634,8331,253
5,0.523966,0.039452,0.508379,0.064140,0.270175,0.746582,0.102040,0.250675,0.373717,0.332772,8331.0,random_forest,cough,all,platt,2957,634,8331,253
6,0.518551,0.036994,0.495628,0.054118,0.161404,0.829853,0.112011,0.269419,0.397924,0.382465,8331.0,random_forest,cough,drop_high_shift,platt,2957,634,8331,232
3,0.518452,0.038638,0.495295,0.057642,0.231579,0.759011,0.110915,0.261331,0.391748,0.358420,8331.0,logistic_regression,cough,top_stable_80,platt,2957,634,8331,80
15,0.518230,0.038589,0.498885,0.059010,0.228070,0.769699,0.098971,0.215036,0.351583,0.298735,8331.0,lightgbm,cough,all,platt,2957,634,8331,253
16,0.515704,0.038170,0.520082,0.069324,0.350877,0.689287,0.116266,0.255569,0.397107,0.312200,8331.0,lightgbm,cough,drop_high_shift,platt,2957,634,8331,232
4,0.515641,0.037084,0.516652,0.068131,0.484211,0.549093,0.110482,0.258040,0.389356,0.286952,8331.0,logistic_regression,cough,top_stable_120,platt,2957,634,8331,120
21,0.513927,0.040939,0.524805,0.071200,0.350877,0.698732,0.118547,0.262631,0.403128,0.352459,8331.0,catboost,cough,drop_high_shift,platt,2957,634,8331,232
0,0.509866,0.037954,0.513102,0.066543,0.315789,0.710415,0.084548,0.202232,0.322501,0.266024,8331.0,logistic_regression,cough,all,platt,2957,634,8331,253



Paper table tail:


,table_source,model_name,modality,fusion_method,dataset,n_samples,auroc,auprc,balanced_accuracy,sensitivity,specificity,f1,brier,ece,nll
20,calibration_metrics,logistic_regression,cough,NaN,NaN,636,0.796,0.672,0.662,0.398,0.926,0.512,0.171,0.100,0.522
21,calibration_metrics,logistic_regression,speech,NaN,NaN,1590,0.748,0.602,0.647,0.400,0.894,0.493,0.184,0.052,0.548
22,calibration_metrics,random_forest,breath,NaN,NaN,636,0.797,0.675,0.661,0.379,0.944,0.506,0.171,0.088,0.522
23,calibration_metrics,random_forest,cough,NaN,NaN,636,0.804,0.717,0.657,0.354,0.960,0.493,0.168,0.090,0.516
24,calibration_metrics,random_forest,speech,NaN,NaN,1590,0.756,0.617,0.652,0.388,0.915,0.496,0.179,0.048,0.538
25,fusion_metrics,NaN,NaN,uniform_mean,NaN,318,0.881,0.841,0.823,0.767,0.879,0.760,0.190,0.153,0.564
26,fusion_metrics,NaN,NaN,validation_weighted_auprc,NaN,318,0.882,0.844,0.813,0.738,0.888,0.749,0.189,0.153,0.564
27,quality_weighted_fusion_metrics,NaN,NaN,quality_weighted_auprc,NaN,318,0.879,0.832,0.804,0.767,0.842,0.731,0.190,0.150,0.564
28,metadata_baseline_metrics,metadata_logistic_regression,metadata,NaN,NaN,2862,0.936,0.922,0.891,0.847,0.934,0.854,0.074,0.075,0.269
29,cross_dataset_metrics,logistic_regression,cough,NaN,NaN,8331,0.529,0.043,0.523,0.232,0.814,0.071,0.151,0.280,0.472


In [6]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent

files = [
  "scripts/25_run_external_model_grid.py",
  "scripts/26_run_coughvid_internal_baseline.py",
  "src/covid_audio_btp/rescue_experiments.py",
  "tests/test_rescue_experiments.py",
]

for f in files:
  p = PROJECT_ROOT / f
  print(f, p.exists())

scripts/25_run_external_model_grid.py True
scripts/26_run_coughvid_internal_baseline.py True
src/covid_audio_btp/rescue_experiments.py True
tests/test_rescue_experiments.py True


In [7]:
import os
import sys
import subprocess

os.chdir(PROJECT_ROOT)

def run_cmd(name, args):
  print(f"\n## {name}")
  print("$", " ".join(str(a) for a in args))
  subprocess.run([str(a) for a in args], check=True)

run_cmd("paper metric tables", [sys.executable, "scripts/20_make_paper_tables.py"])
run_cmd("experiment manifest", [sys.executable, "scripts/24_make_experiment_manifest.py"])
run_cmd("artifact validation", [sys.executable, "scripts/12_validate_artifacts.py", "--strict"])


## paper metric tables
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/20_make_paper_tables.py
Wrote paper metric table: reports/tables/paper_metric_table.csv (60 rows)
Wrote raw combined metric table: reports/tables/paper_metric_table_raw.csv (60 rows)

## experiment manifest
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/24_make_experiment_manifest.py
Wrote experiment manifest: reports/experiment_manifest.json

## artifact validation
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/12_validate_artifacts.py --strict
severity          check                       message
 warning unknown_labels 5688 rows have unknown labels


In [8]:
import pandas as pd

paper = pd.read_csv("reports/tables/paper_metric_table.csv")

print(paper.columns.tolist())

display(
  paper[paper["table_source"].eq("external_model_grid_metrics")]
  [["table_source", "model_name", "modality", "feature_strategy", "calibration_method", "n_samples", "auroc", "auprc", "balanced_accuracy", "f1"]]
  .sort_values(["auroc", "auprc"], ascending=False)
)

['table_source', 'model_name', 'modality', 'feature_strategy', 'fusion_method', 'dataset', 'calibration_method', 'n_samples', 'auroc', 'auprc', 'balanced_accuracy', 'sensitivity', 'specificity', 'f1', 'brier', 'ece', 'nll']


,table_source,model_name,modality,feature_strategy,calibration_method,n_samples,auroc,auprc,balanced_accuracy,f1
32,external_model_grid_metrics,logistic_regression,cough,top_stable_50,platt,8331,0.535,0.042,0.532,0.072
50,external_model_grid_metrics,catboost,cough,all,platt,8331,0.526,0.039,0.517,0.068
35,external_model_grid_metrics,random_forest,cough,all,platt,8331,0.524,0.039,0.508,0.064
36,external_model_grid_metrics,random_forest,cough,drop_high_shift,platt,8331,0.519,0.037,0.496,0.054
33,external_model_grid_metrics,logistic_regression,cough,top_stable_80,platt,8331,0.518,0.039,0.495,0.058
45,external_model_grid_metrics,lightgbm,cough,all,platt,8331,0.518,0.039,0.499,0.059
46,external_model_grid_metrics,lightgbm,cough,drop_high_shift,platt,8331,0.516,0.038,0.520,0.069
34,external_model_grid_metrics,logistic_regression,cough,top_stable_120,platt,8331,0.516,0.037,0.517,0.068
51,external_model_grid_metrics,catboost,cough,drop_high_shift,platt,8331,0.514,0.041,0.525,0.071
30,external_model_grid_metrics,logistic_regression,cough,all,platt,8331,0.510,0.038,0.513,0.067


## Stage 8: Controlled Representation Experiments

This section is disabled by default. Flip only the representation stage you want to run. The intended order is OpenSMILE smoke, OpenSMILE full cough external/internal checks, then one SSL backend at a time.


In [ ]:
# Representation experiment controls. Defaults are safe for Run All.
RUN_OPENSMILE_SMOKE = False
RUN_OPENSMILE_COSWARA_COUGH = False
RUN_OPENSMILE_COUGHVID_COUGH = False
RUN_OPENSMILE_SHIFT = False
RUN_OPENSMILE_EXTERNAL_GRID = False
RUN_OPENSMILE_COUGHVID_INTERNAL = False

RUN_SSL_SMOKE = False
RUN_SSL_COSWARA_COUGH = False
RUN_SSL_COUGHVID_COUGH = False
RUN_SSL_SHIFT = False
RUN_SSL_EXTERNAL_GRID = False
RUN_SSL_COUGHVID_INTERNAL = False

# SSL_BACKEND choices: wav2vec2, beats, panns
SSL_BACKEND = "beats"
SSL_CHECKPOINT_PATH = PROJECT_ROOT / "models/external/beats/model.pt"
SSL_SOURCE_DIR = PROJECT_ROOT / "external/BEATs"

REPRESENTATION_QUALITY_MODE = "quality_ok_only"
REPRESENTATION_MODALITY = "cough"
REPRESENTATION_SMOKE_ROWS = 25
REPRESENTATION_BATCH_SIZE = 8
REPRESENTATION_BOOTSTRAPS = 1000

OPENSMILE_SOURCE_FEATURES = "data/processed/features_opensmile_egemaps_coswara_cough.csv"
OPENSMILE_EXTERNAL_FEATURES = "data/processed/features_opensmile_egemaps_coughvid_cough.csv"
OPENSMILE_SHIFT_REPORT = "reports/tables/feature_shift_opensmile_egemaps_cough.csv"
OPENSMILE_SHIFT_SUMMARY = "reports/tables/feature_shift_opensmile_egemaps_summary.csv"

ssl_slug = SSL_BACKEND.lower().replace("-", "_")
SSL_SOURCE_FEATURES = f"data/processed/features_{ssl_slug}_coswara_cough.csv"
SSL_EXTERNAL_FEATURES = f"data/processed/features_{ssl_slug}_coughvid_cough.csv"
SSL_SHIFT_REPORT = f"reports/tables/feature_shift_{ssl_slug}_cough.csv"
SSL_SHIFT_SUMMARY = f"reports/tables/feature_shift_{ssl_slug}_summary.csv"

print("Representation controls ready")
print("SSL backend:", SSL_BACKEND)
print("SSL checkpoint:", SSL_CHECKPOINT_PATH)
print("SSL source dir:", SSL_SOURCE_DIR)


In [ ]:
# OpenSMILE eGeMAPSv02: stronger handcrafted baseline.
run_step(
    "OpenSMILE Coswara cough smoke",
    [sys.executable, "scripts/27_extract_opensmile_features.py", "--metadata", "data/processed/metadata_clean.csv", "--output", "data/processed/features_opensmile_egemaps_coswara_cough_smoke.csv", "--feature-set", "egemaps", "--quality-mode", REPRESENTATION_QUALITY_MODE, "--modality", REPRESENTATION_MODALITY, "--max-rows", REPRESENTATION_SMOKE_ROWS],
    enabled=RUN_OPENSMILE_SMOKE,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/processed/features_opensmile_egemaps_coswara_cough_smoke.csv"],
    strict_requires=False,
)
run_step(
    "OpenSMILE Coswara cough full",
    [sys.executable, "scripts/27_extract_opensmile_features.py", "--metadata", "data/processed/metadata_clean.csv", "--output", OPENSMILE_SOURCE_FEATURES, "--feature-set", "egemaps", "--quality-mode", REPRESENTATION_QUALITY_MODE, "--modality", REPRESENTATION_MODALITY],
    enabled=RUN_OPENSMILE_COSWARA_COUGH,
    requires=["data/processed/metadata_clean.csv"],
    creates=[OPENSMILE_SOURCE_FEATURES],
    strict_requires=False,
)
run_step(
    "OpenSMILE COUGHVID cough full",
    [sys.executable, "scripts/27_extract_opensmile_features.py", "--metadata", "data/interim/coughvid_index.csv", "--output", OPENSMILE_EXTERNAL_FEATURES, "--feature-set", "egemaps", "--quality-mode", REPRESENTATION_QUALITY_MODE, "--modality", REPRESENTATION_MODALITY, "--split-name", "external"],
    enabled=RUN_OPENSMILE_COUGHVID_COUGH,
    requires=["data/interim/coughvid_index.csv"],
    creates=[OPENSMILE_EXTERNAL_FEATURES],
    strict_requires=False,
)
run_step(
    "OpenSMILE feature shift",
    [sys.executable, "scripts/23_feature_shift_report.py", "--source-features", OPENSMILE_SOURCE_FEATURES, "--external-features", OPENSMILE_EXTERNAL_FEATURES, "--output", OPENSMILE_SHIFT_REPORT, "--summary-output", OPENSMILE_SHIFT_SUMMARY],
    enabled=RUN_OPENSMILE_SHIFT,
    requires=[OPENSMILE_SOURCE_FEATURES, OPENSMILE_EXTERNAL_FEATURES],
    creates=[OPENSMILE_SHIFT_REPORT, OPENSMILE_SHIFT_SUMMARY],
    strict_requires=False,
)
run_step(
    "OpenSMILE external model grid",
    [sys.executable, "scripts/25_run_external_model_grid.py", "--source-features", OPENSMILE_SOURCE_FEATURES, "--external-features", OPENSMILE_EXTERNAL_FEATURES, "--feature-shift-report", OPENSMILE_SHIFT_REPORT, "--predictions-output", "data/outputs/metrics/external_model_grid_opensmile_egemaps_predictions.csv", "--metrics-output", "data/outputs/metrics/external_model_grid_opensmile_egemaps_metrics.csv", "--bootstrap-output", "data/outputs/metrics/external_model_grid_opensmile_egemaps_bootstrap_ci.csv", "--n-bootstraps", REPRESENTATION_BOOTSTRAPS],
    enabled=RUN_OPENSMILE_EXTERNAL_GRID,
    requires=[OPENSMILE_SOURCE_FEATURES, OPENSMILE_EXTERNAL_FEATURES, OPENSMILE_SHIFT_REPORT],
    creates=["data/outputs/metrics/external_model_grid_opensmile_egemaps_metrics.csv"],
    strict_requires=False,
)
run_step(
    "OpenSMILE COUGHVID internal baseline",
    [sys.executable, "scripts/26_run_coughvid_internal_baseline.py", "--features", OPENSMILE_EXTERNAL_FEATURES, "--split-features-output", "data/processed/features_opensmile_egemaps_coughvid_internal_split.csv", "--predictions-output", "data/outputs/metrics/coughvid_internal_opensmile_egemaps_predictions.csv", "--metrics-output", "data/outputs/metrics/coughvid_internal_opensmile_egemaps_metrics.csv", "--bootstrap-output", "data/outputs/metrics/coughvid_internal_opensmile_egemaps_bootstrap_ci.csv", "--n-bootstraps", REPRESENTATION_BOOTSTRAPS],
    enabled=RUN_OPENSMILE_COUGHVID_INTERNAL,
    requires=[OPENSMILE_EXTERNAL_FEATURES],
    creates=["data/outputs/metrics/coughvid_internal_opensmile_egemaps_metrics.csv"],
    strict_requires=False,
)


In [ ]:
# SSL embeddings: run exactly one backend at a time.
def ssl_embedding_cmd(metadata, output, *, split_name=None, max_rows=None):
    cmd = [sys.executable, "scripts/28_extract_ssl_embeddings.py", "--metadata", metadata, "--output", output, "--backend", SSL_BACKEND, "--quality-mode", REPRESENTATION_QUALITY_MODE, "--modality", REPRESENTATION_MODALITY, "--batch-size", REPRESENTATION_BATCH_SIZE]
    if SSL_BACKEND.lower().replace("-", "_") in {"beats", "beats_official", "panns", "panns_cnn14", "panns_cnn14_official"}:
        cmd += ["--checkpoint-path", SSL_CHECKPOINT_PATH, "--source-dir", SSL_SOURCE_DIR]
    if split_name is not None:
        cmd += ["--split-name", split_name]
    if max_rows is not None:
        cmd += ["--max-rows", max_rows]
    return cmd

ssl_model_requires = []
if SSL_BACKEND.lower().replace("-", "_") in {"beats", "beats_official", "panns", "panns_cnn14", "panns_cnn14_official"}:
    ssl_model_requires = [SSL_CHECKPOINT_PATH, SSL_SOURCE_DIR]

run_step(
    f"{SSL_BACKEND} Coswara cough smoke",
    ssl_embedding_cmd("data/processed/metadata_clean.csv", f"data/processed/features_{ssl_slug}_coswara_cough_smoke.csv", max_rows=REPRESENTATION_SMOKE_ROWS),
    enabled=RUN_SSL_SMOKE,
    requires=["data/processed/metadata_clean.csv", *ssl_model_requires],
    creates=[f"data/processed/features_{ssl_slug}_coswara_cough_smoke.csv"],
    strict_requires=False,
)
run_step(
    f"{SSL_BACKEND} Coswara cough full",
    ssl_embedding_cmd("data/processed/metadata_clean.csv", SSL_SOURCE_FEATURES),
    enabled=RUN_SSL_COSWARA_COUGH,
    requires=["data/processed/metadata_clean.csv", *ssl_model_requires],
    creates=[SSL_SOURCE_FEATURES],
    strict_requires=False,
)
run_step(
    f"{SSL_BACKEND} COUGHVID cough full",
    ssl_embedding_cmd("data/interim/coughvid_index.csv", SSL_EXTERNAL_FEATURES, split_name="external"),
    enabled=RUN_SSL_COUGHVID_COUGH,
    requires=["data/interim/coughvid_index.csv", *ssl_model_requires],
    creates=[SSL_EXTERNAL_FEATURES],
    strict_requires=False,
)
run_step(
    f"{SSL_BACKEND} feature shift",
    [sys.executable, "scripts/23_feature_shift_report.py", "--source-features", SSL_SOURCE_FEATURES, "--external-features", SSL_EXTERNAL_FEATURES, "--output", SSL_SHIFT_REPORT, "--summary-output", SSL_SHIFT_SUMMARY],
    enabled=RUN_SSL_SHIFT,
    requires=[SSL_SOURCE_FEATURES, SSL_EXTERNAL_FEATURES],
    creates=[SSL_SHIFT_REPORT, SSL_SHIFT_SUMMARY],
    strict_requires=False,
)
run_step(
    f"{SSL_BACKEND} external model grid",
    [sys.executable, "scripts/25_run_external_model_grid.py", "--source-features", SSL_SOURCE_FEATURES, "--external-features", SSL_EXTERNAL_FEATURES, "--feature-shift-report", SSL_SHIFT_REPORT, "--predictions-output", f"data/outputs/metrics/external_model_grid_{ssl_slug}_predictions.csv", "--metrics-output", f"data/outputs/metrics/external_model_grid_{ssl_slug}_metrics.csv", "--bootstrap-output", f"data/outputs/metrics/external_model_grid_{ssl_slug}_bootstrap_ci.csv", "--n-bootstraps", REPRESENTATION_BOOTSTRAPS],
    enabled=RUN_SSL_EXTERNAL_GRID,
    requires=[SSL_SOURCE_FEATURES, SSL_EXTERNAL_FEATURES, SSL_SHIFT_REPORT],
    creates=[f"data/outputs/metrics/external_model_grid_{ssl_slug}_metrics.csv"],
    strict_requires=False,
)
run_step(
    f"{SSL_BACKEND} COUGHVID internal baseline",
    [sys.executable, "scripts/26_run_coughvid_internal_baseline.py", "--features", SSL_EXTERNAL_FEATURES, "--split-features-output", f"data/processed/features_{ssl_slug}_coughvid_internal_split.csv", "--predictions-output", f"data/outputs/metrics/coughvid_internal_{ssl_slug}_predictions.csv", "--metrics-output", f"data/outputs/metrics/coughvid_internal_{ssl_slug}_metrics.csv", "--bootstrap-output", f"data/outputs/metrics/coughvid_internal_{ssl_slug}_bootstrap_ci.csv", "--n-bootstraps", REPRESENTATION_BOOTSTRAPS],
    enabled=RUN_SSL_COUGHVID_INTERNAL,
    requires=[SSL_EXTERNAL_FEATURES],
    creates=[f"data/outputs/metrics/coughvid_internal_{ssl_slug}_metrics.csv"],
    strict_requires=False,
)
